# nlvms Colab runner
Run top to bottom. Every cell is safe to re-run. Set `WHO`, `RUN` and `GITHUB_REPO` in the config cell.

In [ ]:
# 1. GPU check (Runtime > Change runtime type > T4 GPU)
!nvidia-smi
import torch
print(torch.__version__, "cuda:", torch.cuda.is_available())

In [ ]:
# 2. Mount Drive and get the code. Needs a Colab Secret named GH_TOKEN
#    (key icon in the left sidebar, enable notebook access).
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
GITHUB_REPO = "your-username/nlvms"          # <- edit
token = userdata.get('GH_TOKEN')
if not os.path.exists('/content/nlvms'):
    !git clone https://{token}@github.com/{GITHUB_REPO}.git /content/nlvms
%cd /content/nlvms
!git pull

In [ ]:
# 3. Dependencies (do not reinstall torch)
!pip install -q einops omegaconf spacy nltk
!python -m spacy download en_core_web_sm

In [ ]:
# 4. OPTIONAL fast Mamba kernel. Skip on day one: the PyTorch fallback works.
#    Can take ~40 min and may fail on version mismatch. Do it in its own session,
#    then restart the runtime and rerun cells 1-3.
# !pip install -q causal-conv1d mamba-ssm --no-build-isolation

In [ ]:
# 5. Identity, config, lock, checkpoints
import sys
sys.path.insert(0, 'src')
from utils.colab import load_config, RunLock, flush_drive
from utils.checkpoint import CheckpointManager

WHO = "diksha"          # <- each person sets their own name
RUN = "qvh_baseline"    # <- one folder per experiment

cfg = load_config("configs/default.yaml", drive_root="/content/drive/MyDrive/nlvms")
lock = RunLock(f"{cfg.paths.drive_root}/locks/{RUN}.json", who=WHO)
lock.acquire()
ckpt = CheckpointManager(f"{cfg.paths.checkpoints}/{RUN}", keep_last=2)
print(cfg.paths)

In [ ]:
# 6. Copy features from Drive to local disk (Drive reads are slow; later runs are quick)
!mkdir -p /content/features
!rsync -a /content/drive/MyDrive/nlvms/features/ /content/features/
cfg.data.feature_dir = "/content/features"

In [ ]:
# 7. Environment smoke tests on the GPU
!python tests/smoke_mamba.py
!python tests/test_checkpoint.py
!python tests/test_config.py

### Cross-account resume dry-run (toy model)
Account A runs the first command, then cell 9. Account B runs cells 1-3 and 5, then the second command.

In [ ]:
# 8a. Account A: trains epochs 0-2, then stops
!python scripts/dryrun_resume.py --who diksha

In [ ]:
# 8b. Account B (change --who): should print "resumed from epoch_002.pt" and finish epochs 3-5
# !python scripts/dryrun_resume.py --who bhavagna

In [ ]:
# 9. HAND OFF: run before stopping. Releases the lock and flushes Drive writes.
lock.release()
flush_drive()